In [1]:
%reload_ext dotenv

In [2]:
from pathlib import Path
from dotenv import load_dotenv
env_file = Path("/Users/lvalverdeb/TeamDev/repo-split/boti-data/.env.local")
load_dotenv(dotenv_path=env_file)

True

In [3]:
import os
db_url_async = os.getenv('ASYNC_DB_DSN')
db_url_sync = os.getenv('SYNC_DB_DSN')

In [4]:
from boti_data.helper import DataHelper
config={
    'backend': 'sqlalchemy',
    'connection_url': db_url_async,
    "worker_connection_env_var": "ASYNC_DB_DSN",
    "poolclass": "sqlalchemy.pool.NullPool",
    'query_only': True,
    'table': 'asm_tracking_productos',
    'field_map': {
        'id_track_global': 'global_track_id',
        'id_tipo_producto': 'product_type_id'
    },
    'sticky_filters': {
        'product_type_id': 1,
    },
}

In [5]:
gateway = DataHelper(**config)
columns=['id_producto','cliente_id', 'product_type_id','global_track_id']

In [6]:
result_pandas = await gateway.pandas.aload(global_track_id__in=[5],columns=columns)


In [7]:
result_pandas

,id_producto,cliente_id,product_type_id,global_track_id
0,405719380,231,1,5
1,405719450,209,1,5
2,405719451,157,1,5
3,405719452,209,1,5
4,405719453,209,1,5
...,...,...,...,...
1718055,408528983,324,1,5
1718056,408528985,324,1,5
1718057,408529290,328,1,5
1718058,408529292,328,1,5


In [8]:
result_polars = await gateway.polars.aload(global_track_id__in=[5], columns=columns)

In [9]:
result_polars

id_producto,cliente_id,product_type_id,global_track_id
i32,i32,i32,i32
405719380,231,1,5
405719450,209,1,5
405719451,157,1,5
405719452,209,1,5
405719453,209,1,5
…,…,…,…
408528983,324,1,5
408528985,324,1,5
408529290,328,1,5


In [10]:
import time
t0 = time.time()
result_dask = await gateway.dask.aload(global_track_id__in=[5], columns=columns)
t1 = time.time()
print(f"dask aload: {t1-t0:.2f}s")

dask aload: 6.38s


In [11]:
result_dask.compute()
#t2 = time.time()
#print(f"compute: {t2-t1:.2f}s")

,id_producto,cliente_id,product_type_id,global_track_id
0,405719380,231,1,5
1,405719450,209,1,5
2,405719451,157,1,5
3,405719452,209,1,5
4,405719453,209,1,5
...,...,...,...,...
1718055,408528983,324,1,5
1718056,408528985,324,1,5
1718057,408529290,328,1,5
1718058,408529292,328,1,5


In [12]:
import time
# Re-run with diagnostics=True to see sub-step timing
t0 = time.time()
result_dask_diag = await gateway.dask.aload(
    global_track_id__in=[1,2,3,4],
    columns=columns,
    diagnostics=True,
)
t1 = time.time()
print(f"dask aload (diagnostics): {t1-t0:.2f}s")


[2026-07-13 17:32:24][INFO][AsyncSqlDatabaseResource] Configured async select reflect_select=0.000s
[2026-07-13 17:32:24][INFO][AsyncSqlDatabaseResource] Build partitioned request elapsed=0.000s
[2026-07-13 17:32:27][INFO][AsyncSqlDatabaseResource] Partitioned SQL fast path: single partition rows=15633 ceiling=5000000 est_bytes_per_row=32 chunk_size=50000 total=3.192s planner_init=0.000s prepare_stmt=0.000s db_fetch=3.189s coerce=0.001s from_pandas=0.001s
dask aload (diagnostics): 3.29s


In [13]:
import time
# Compare with sync DSN (mysql://) to check if conn.run_sync bridging adds overhead
# Note: sync DSN path uses SqlPartitionedLoader.load_request() directly (no run_sync wrapping)
sync_config = dict(config)
sync_config["connection_url"] = db_url_sync
sync_config["worker_connection_env_var"] = "SYNC_DB_DSN"
sync_gateway = DataHelper(**sync_config)
t0 = time.time()
result_sync = await sync_gateway.dask.aload(
    global_track_id__in=[5],
    columns=columns,
    diagnostics=True,
)
t1 = time.time()
print(f"sync dask aload: {t1-t0:.2f}s")
pdf_sync = result_sync.compute()
t2 = time.time()
print(f"sync compute: {t2-t1:.2f}s  rows={len(pdf_sync)}")


[2026-07-13 17:32:28][INFO][SqlDatabaseResource] Gateway load starting backend=sqlalchemy configured=True requested_return_type=dask resolved_return_type=dask requested_execution_mode=lazy resolved_execution_mode=lazy loader_return_type=dask persist=False
[2026-07-13 17:32:32][INFO][SqlDatabaseResource] Partitioned SQL plan strategy=offset partitions=35 rows=1718060 chunk_size=50000 max_concurrent_fetches=4 use_arrow=False
[2026-07-13 17:32:32][INFO][SqlDatabaseResource] Partitioned SQL load completed strategy=offset partitions=35 rows=1718060 result_partitions=35 elapsed=4.40s prepare_stmt=0.000s load_plan=0.005s
[2026-07-13 17:32:32][WARNING][SqlDatabaseResource] SqlPartitionedLoader was garbage collected without being closed. Resources may have leaked. Please use context managers or call .close().
sync dask aload: 4.68s
[2026-07-13 17:32:32][INFO][SqlDatabaseResource] Gateway load completed elapsed=4.67s metrics={'engine': 'dask', 'columns': 4, 'npartitions': 35, 'graph_tasks': 175,

## SQL Advanced Filter Tests

Testing all filter operator types against the SQL backend via `DataHelper`.
Covers equality, comparison, range, string patterns, null checks, and boolean logic.

In [14]:
print("=" * 72)
print("SQL FILTER TESTS")
print("=" * 72)

# 1. Single equality via mapped field (global_track_id → id_track_global)
result = await gateway.pandas.aload(global_track_id=1, columns=["id_producto", "global_track_id"])
print(f"global_track_id=1: rows={len(result)}, cols={list(result.columns)}")
assert len(result) > 0
assert (result["global_track_id"] == 1).all()
print("  ✓ equality filter works with field_map translation")
print()


SQL FILTER TESTS
global_track_id=1: rows=12003, cols=['id_producto', 'global_track_id']
  ✓ equality filter works with field_map translation



In [15]:
# 2. Greater-than filters
result = await gateway.pandas.aload(id_producto__gte=408000000, columns=["id_producto"])
print(f"id_producto__gte=408000000: rows={len(result)}, min={result["id_producto"].min()}")
assert len(result) > 0
assert result["id_producto"].min() >= 408000000
print("  ✓ __gte filter works")

result = await gateway.pandas.aload(id_producto__lt=405730000, columns=["id_producto"])
print(f"id_producto__lt=405730000: rows={len(result)}, max={result["id_producto"].max()}")
assert len(result) > 0
assert result["id_producto"].max() < 405730000
print("  ✓ __lt filter works")
print()


id_producto__gte=408000000: rows=495733, min=408000000
  ✓ __gte filter works
id_producto__lt=405730000: rows=3316, max=405729515
  ✓ __lt filter works



In [16]:
# 3. Range (between) — rewritten to __gte + __lte
result = await gateway.pandas.aload(id_producto__range=(405780000, 405790000), columns=["id_producto"])
print(f"id_producto__range=(405780000, 405790000): rows={len(result)}")
assert len(result) > 0
assert result["id_producto"].between(405780000, 405790000).all()
print("  ✓ __range filter works")
print()


id_producto__range=(405780000, 405790000): rows=3933
  ✓ __range filter works



In [17]:
# 4. Not equal and not_in
result = await gateway.pandas.aload(global_track_id__ne=1, columns=["global_track_id"])
print(f"global_track_id__ne=1: rows={len(result)}")
if len(result) > 0:
    assert (result["global_track_id"] != 1).all()
print("  ✓ __ne filter works")

result = await gateway.pandas.aload(global_track_id__not_in=[2, 3, 4], columns=["global_track_id"])
print(f"global_track_id__not_in=[2,3,4]: rows={len(result)}")
assert len(result) > 0
print("  ✓ __not_in filter works")
print()


global_track_id__ne=1: rows=1721690
  ✓ __ne filter works
global_track_id__not_in=[2,3,4]: rows=1730063
  ✓ __not_in filter works



In [18]:
# 5. String pattern filters (residual — applied after SQL fetch)
# Note: MySQL LIKE is case-insensitive by default
import pandas as pd
result = await gateway.pandas.aload(codigo_barra__startswith="IST", columns=["codigo_barra"])
print(f"codigo_barra__startswith=IST: rows={len(result)}")
if len(result) > 0:
    assert all(str(v).lower().startswith("ist") for v in result["codigo_barra"] if pd.notna(v))
print("  ✓ __startswith filter works")

result = await gateway.pandas.aload(nombre_th__contains="J", columns=["nombre_th"])
print(f"nombre_th__contains=J: rows={len(result)}")
if len(result) > 0:
    assert all("J" in str(v).upper() for v in result["nombre_th"] if pd.notna(v))
print("  ✓ __contains filter works")

result = await gateway.pandas.aload(codigo_barra__endswith="CR", columns=["codigo_barra"])
print(f"codigo_barra__endswith=CR: rows={len(result)}")
if len(result) > 0:
    # MySQL LIKE is case-insensitive, so match lower-case
    assert all(str(v).lower().endswith("cr") for v in result["codigo_barra"] if pd.notna(v))
print("  ✓ __endswith filter works")
print()


codigo_barra__startswith=IST: rows=6844
  ✓ __startswith filter works
nombre_th__contains=J: rows=344501
  ✓ __contains filter works
codigo_barra__endswith=CR: rows=38609
  ✓ __endswith filter works



In [19]:
# 6. Null checks
result_not_null = await gateway.pandas.aload(sub_estatus__isnull=False, columns=["sub_estatus"])
print(f"sub_estatus IS NOT NULL: rows={len(result_not_null)}")

result_null = await gateway.pandas.aload(sub_estatus__isnull=True, columns=["sub_estatus"])
print(f"sub_estatus IS NULL: rows={len(result_null)}")
print("  ✓ __isnull filter works (both True and False)")
print()


sub_estatus IS NOT NULL: rows=1700439
sub_estatus IS NULL: rows=33254
  ✓ __isnull filter works (both True and False)



In [20]:
# 7. Composite filters (implicit AND)
result = await gateway.pandas.aload(cliente_id=139, id_producto__gte=408000000, columns=["cliente_id", "id_producto"])
print(f"cliente_id=139 AND id_producto>=408000000: rows={len(result)}")
assert len(result) > 0
assert (result["cliente_id"] == 139).all()
assert (result["id_producto"] >= 408000000).all()
print("  ✓ Multiple filters implicitly ANDed")
print()


cliente_id=139 AND id_producto>=408000000: rows=217
  ✓ Multiple filters implicitly ANDed



In [21]:
# 8. Explicit OR
result = await gateway.pandas.aload(filters={"$or": [{"cliente_id": 139}, {"cliente_id": 209}]}, columns=["cliente_id"])
print(f"$or: cliente_id=139 OR 209: rows={len(result)}")
assert len(result) > 0
assert result["cliente_id"].isin([139, 209]).all()
print("  ✓ $or boolean filter works")
print()


$or: cliente_id=139 OR 209: rows=95265
  ✓ $or boolean filter works



In [22]:
# 9. IN filter
result = await gateway.pandas.aload(cliente_id__in=[139, 209, 304], columns=["cliente_id"])
print(f"cliente_id__in=[139,209,304]: rows={len(result)}")
assert len(result) > 0
assert set(result["cliente_id"].unique()).issubset({139, 209, 304})
print("  ✓ __in filter works")
print()


cliente_id__in=[139,209,304]: rows=99071
  ✓ __in filter works



In [23]:
print("All SQL filter tests passed.")


All SQL filter tests passed.


In [24]:
from boti_data.connection_catalog import S3Catalog
from boti.core.filesystem import add_endpoint_to_allowlist
add_endpoint_to_allowlist(os.getenv('ETL_ENDPOINT_ALLOWLIST'))
store = S3Catalog("ETL_", env_file=".env")
#add_endpoint_to_allowlist(store.endpoint_allowlist)
print(store.storage_path)   # s3://analytics-bucket/raw/events
print(store.ls())

s3://dst-etl
['dst-etl/bronze']


In [25]:
store

In [26]:
from boti_data import ParquetReader

In [27]:
parquet_config = {
    'fs': store.fs(),
    'storage_path': 'dst-etl/bronze/logistics/mobile/gps/',
    'parquet_start_date': '2025-01-01',
    'parquet_end_date': '2026-03-31',
    'partition_on': ['partition_date']
}
parquet_helper = ParquetReader(**parquet_config)

In [28]:
result = await parquet_helper.aload(associate_id__in=[27,2285])

In [29]:
result.compute()

,id,product_id,date_time,associate_id,latitude,longitude,direccion,action,description,server_dt,proveedor_gps,imei,android_id,partition_date
0,717299351,-1,2026-01-01 00:19:21+00:00,2285,9.965745,-84.065721,<NA>,Reporte Posición,Monitoreo Regular,2026-01-01 07:50:03+00:00,fused,861275086038987,d72aa21320c21380,2026-01-01
1,717299352,-1,2026-01-01 00:44:23+00:00,2285,9.965712,-84.065716,<NA>,Reporte Posición,Monitoreo Regular,2026-01-01 07:50:04+00:00,fused,861275086038987,d72aa21320c21380,2026-01-01
2,717299353,-1,2026-01-01 01:14:59+00:00,2285,9.965729,-84.065702,<NA>,Reporte Posición,Monitoreo Regular,2026-01-01 07:50:04+00:00,fused,861275086038987,d72aa21320c21380,2026-01-01
3,717299354,-1,2026-01-01 01:39:16+00:00,2285,9.965733,-84.065724,<NA>,Reporte Posición,Monitoreo Regular,2026-01-01 07:50:04+00:00,fused,861275086038987,d72aa21320c21380,2026-01-01
4,717299355,-1,2026-01-01 02:09:32+00:00,2285,9.965763,-84.065715,<NA>,Reporte Posición,Monitoreo Regular,2026-01-01 07:50:04+00:00,fused,861275086038987,d72aa21320c21380,2026-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20903,718852114,-1,2026-03-31 22:52:30+00:00,2285,9.96577,-84.065751,<NA>,Reporte Posición,Monitoreo Regular,2026-03-31 22:52:38+00:00,fused,861275086038987,d72aa21320c21380,2026-03-31
20904,718852115,-1,2026-03-31 22:52:35+00:00,2285,9.965823,-84.065731,<NA>,Reporte Posición,Monitoreo Regular,2026-03-31 22:52:38+00:00,fused,861275086038987,d72aa21320c21380,2026-03-31
20905,718852260,-1,2026-03-31 22:59:44+00:00,2285,9.965819,-84.065734,<NA>,Reporte Posición,Monitoreo Regular,2026-04-01 00:18:17+00:00,fused,861275086038987,d72aa21320c21380,2026-03-31
20906,718852261,-1,2026-03-31 23:17:53+00:00,2285,9.965777,-84.065731,<NA>,Reporte Posición,Monitoreo Regular,2026-04-01 00:18:17+00:00,fused,861275086038987,d72aa21320c21380,2026-03-31


In [30]:
parquet_helper.close()

## Parquet Advanced Filter Tests

Testing all filter types against the Parquet (S3) backend.
Operators are classified as **pushdown** (pruned at scan time by PyArrow) or **residual**
(applied in-memory via Arrow compute kernels).

In [31]:
# Recreate ParquetReader for comprehensive filter tests
# Using a fresh config to avoid mutating the original helper
from boti_data import DataHelper

parquet_config = {
    'backend': 'parquet',
    'fs': store.fs(),
    'storage_path': 'dst-etl/bronze/logistics/mobile/gps/',
    'parquet_start_date': '2026-01-01',
    'parquet_end_date': '2026-03-31',
    'partition_on': ['partition_date'],
}
parquet = DataHelper(**parquet_config)
print(f"Parquet filter test helper ready: {parquet_config['storage_path']}")


Parquet filter test helper ready: dst-etl/bronze/logistics/mobile/gps/


In [32]:
print("=" * 72)
print("PARQUET BASIC (PUSHDOWN) FILTER TESTS")
print("=" * 72)

# 1. Equality filter (exact)
result = await parquet.aload(associate_id=2285, columns=["associate_id", "action", "latitude"])
pdf = result.compute()
print(f"associate_id=2285: rows={len(pdf)}")
assert len(pdf) > 0
assert (pdf["associate_id"] == 2285).all()
print("  ✓ __exact filter works (pushdown)")
print()


PARQUET BASIC (PUSHDOWN) FILTER TESTS
associate_id=2285: rows=16926
  ✓ __exact filter works (pushdown)



In [33]:
# 2. Comparison filters
result = await parquet.aload(associate_id__gte=4000, columns=["associate_id", "action"])
pdf = result.compute()
print(f"associate_id__gte=4000: rows={len(pdf)}, min={pdf["associate_id"].min()}")
assert len(pdf) > 0
assert pdf["associate_id"].min() >= 4000
print("  ✓ __gte pushes down to PyArrow scan")

result = await parquet.aload(associate_id__lt=100, columns=["associate_id"])
pdf = result.compute()
print(f"associate_id__lt=100: rows={len(pdf)}, max={pdf["associate_id"].max()}")
assert len(pdf) > 0
assert pdf["associate_id"].max() < 100
print("  ✓ __lt filter works (pushdown)")
print()


associate_id__gte=4000: rows=605235, min=4005
  ✓ __gte pushes down to PyArrow scan
associate_id__lt=100: rows=3982, max=27
  ✓ __lt filter works (pushdown)



In [34]:
# 3. Range and NOT filters
result = await parquet.aload(associate_id__range=(100, 5000), columns=["associate_id"])
pdf = result.compute()
print(f"associate_id__range=(100, 5000): rows={len(pdf)}")
assert len(pdf) > 0
assert pdf["associate_id"].between(100, 5000).all()
print("  ✓ __range filter works (rewritten to gte+lte)")

result = await parquet.aload(associate_id__not_exact=2285, columns=["associate_id"])
pdf = result.compute()
print(f"associate_id__ne=2285: rows={len(pdf)}")
assert len(pdf) > 0
assert (pdf["associate_id"] != 2285).all()
print("  ✓ __ne filter works (pushdown)")

result = await parquet.aload(associate_id__not_in=[2285, 4499], columns=["associate_id"])
pdf = result.compute()
print(f"associate_id__not_in=[2285,4499]: rows={len(pdf)}")
assert len(pdf) > 0
assert not pdf["associate_id"].isin([2285, 4499]).any()
print("  ✓ __not_in filter works (pushdown)")
print()


associate_id__range=(100, 5000): rows=1385129
  ✓ __range filter works (rewritten to gte+lte)
associate_id__ne=2285: rows=1397995
  ✓ __ne filter works (pushdown)
associate_id__not_in=[2285,4499]: rows=1390184
  ✓ __not_in filter works (pushdown)



In [35]:
# 4. Float column filters
result = await parquet.aload(latitude__gte=9.90, columns=["latitude", "longitude", "associate_id"])
pdf = result.compute()
print(f"latitude__gte=9.90: rows={len(pdf)}")
assert len(pdf) > 0
assert pdf["latitude"].min() >= 9.90
print("  ✓ Float column filter works (pushdown)")

result = await parquet.aload(longitude__lte=-84.0, columns=["longitude"])
pdf = result.compute()
print(f"longitude__lte=-84.0: rows={len(pdf)}")
if len(pdf) > 0:
    assert (pdf["longitude"] <= -84.0).all()
print("  ✓ Float __lte filter works (pushdown)")
print()


latitude__gte=9.90: rows=1117820
  ✓ Float column filter works (pushdown)
longitude__lte=-84.0: rows=1136659
  ✓ Float __lte filter works (pushdown)



### Parquet Residual (Post-Scan) Filter Tests

Operators like `__startswith`, `__contains`, `__isnull` cannot be pushed into the
Parquet scan. They are applied in-memory via **Arrow compute kernels** after the
relevant row groups are loaded.

In [36]:
print("=" * 72)
print("PARQUET RESIDUAL FILTER TESTS")
print("=" * 72)

# 5. String pattern — startswith (residual for Arrow)
result = await parquet.aload(action__startswith="Acceso", columns=["action", "associate_id"])
pdf = result.compute()
print(f"action__startswith=Acceso: rows={len(pdf)}")
assert len(pdf) > 0
assert all(str(v).startswith("Acceso") for v in pdf["action"])
print("  ✓ __startswith applied via Arrow residual")

# 6. String contains
result = await parquet.aload(description__contains="Regular", columns=["description"])
pdf = result.compute()
print(f"description__contains=Regular: rows={len(pdf)}")
assert len(pdf) > 0
assert all("Regular" in str(v) for v in pdf["description"])
print("  ✓ __contains applied via Arrow residual")

# 7. String endswith
result = await parquet.aload(action__endswith="Salida", columns=["action"])
pdf = result.compute()
print(f"action__endswith=Salida: rows={len(pdf)}")
if len(pdf) > 0:
    assert all(str(v).endswith("Salida") for v in pdf["action"])
print("  ✓ __endswith applied via Arrow residual")
print()


PARQUET RESIDUAL FILTER TESTS
action__startswith=Acceso: rows=23353
  ✓ __startswith applied via Arrow residual
description__contains=Regular: rows=900621
  ✓ __contains applied via Arrow residual
action__endswith=Salida: rows=23353
  ✓ __endswith applied via Arrow residual



In [37]:
# 8. Null checks
result = await parquet.aload(direccion__isnull=True, columns=["direccion", "action"])
pdf = result.compute()
print(f"direccion IS NULL: rows={len(pdf)}")
assert len(pdf) > 0
assert pdf["direccion"].isna().all()
print("  ✓ __isnull=True works (residual)")

# direccion is mostly null; verify not-null returns 0 or a valid set
result = await parquet.aload(direccion__isnull=False, columns=["direccion"])
pdf = result.compute()
print(f"direccion IS NOT NULL: rows={len(pdf)}")
if len(pdf) > 0:
    assert pdf["direccion"].notna().all()
print("  ✓ __isnull=False works (residual)")
print()


direccion IS NULL: rows=1414921
  ✓ __isnull=True works (residual)
direccion IS NOT NULL: rows=0
  ✓ __isnull=False works (residual)



In [38]:
# 9. Case-insensitive filters (residual)
result = await parquet.aload(action__icontains="acceso", columns=["action"])
pdf = result.compute()
print(f"action__icontains=acceso: rows={len(pdf)}")
assert len(pdf) > 0
assert all("acceso" in str(v).lower() for v in pdf["action"])
print("  ✓ __icontains works (case-insensitive residual)")

# istartswith
result = await parquet.aload(action__istartswith="reporte", columns=["action"])
pdf = result.compute()
print(f"action__istartswith=reporte: rows={len(pdf)}")
assert len(pdf) > 0
assert all(str(v).lower().startswith("reporte") for v in pdf["action"])
print("  ✓ __istartswith works (case-insensitive residual)")
print()


action__icontains=acceso: rows=23353
  ✓ __icontains works (case-insensitive residual)
action__istartswith=reporte: rows=900621
  ✓ __istartswith works (case-insensitive residual)



In [39]:
print("=" * 72)
print("PARQUET COMPOSITE FILTER TESTS")
print("=" * 72)

# 10. Implicit AND — pushdown + residual combined
result = await parquet.aload(
    associate_id__gte=4000,
    action__startswith="Reporte",
    columns=["associate_id", "action"],
)
pdf = result.compute()
print(f"associate_id>=4000 AND action starts with Reporte: rows={len(pdf)}")
assert len(pdf) > 0
assert (pdf["associate_id"] >= 4000).all()
assert all(str(v).startswith("Reporte") for v in pdf["action"])
print("  ✓ Pushdown + residual filters compose correctly")
print()


PARQUET COMPOSITE FILTER TESTS
associate_id>=4000 AND action starts with Reporte: rows=410165
  ✓ Pushdown + residual filters compose correctly



In [40]:
# 11. Column projection + filter
result = await parquet.aload(
    associate_id__gte=4000,
    columns=["associate_id", "action", "latitude"],
)
pdf = result.compute()
print(f"Projected columns: {list(pdf.columns)}")
assert list(pdf.columns) == ["associate_id", "action", "latitude"]
assert (pdf["associate_id"] >= 4000).all()
print("  ✓ Column projection + filter works together")
print()


Projected columns: ['associate_id', 'action', 'latitude']
  ✓ Column projection + filter works together



In [41]:
# 12. Explicit OR filter
result = await parquet.aload(filters={
    "$or": [
        {"associate_id": 27},
        {"associate_id": 4499},
    ]
}, columns=["associate_id", "action"])
pdf = result.compute()
print(f"$or: associate_id=27 OR 4499: rows={len(pdf)}")
assert len(pdf) > 0
assert pdf["associate_id"].isin([27, 4499]).all()
print("  ✓ $or boolean filter works on Parquet backend")
print()


$or: associate_id=27 OR 4499: rows=11793
  ✓ $or boolean filter works on Parquet backend



In [42]:
print("All Parquet filter tests passed.")


All Parquet filter tests passed.


In [43]:
from boti_dask import (
    UniqueValuesExtractor,
    apply_recommended_dask_config,
    async_safe_compute,
    async_safe_gather,
    async_safe_head,
    async_safe_persist,
    async_safe_wait,
    dask_is_empty,
    dask_is_probably_empty,
    inspect_graph,
    safe_compute,
    safe_gather,
    safe_head,
    safe_persist,
    safe_wait,
)
import dask

graph_metrics = inspect_graph(result_dask)
assert graph_metrics["is_dask"] is True
assert graph_metrics["npartitions"] == result_dask.npartitions

with apply_recommended_dask_config():
    assert dask.config.get("dataframe.shuffle.method") == "tasks"

graph_metrics

{'type': 'DataFrame',
 'is_dask': True,
 'task_count': 3,
 'npartitions': 1,
 'graph_layers': None}

In [44]:
with gateway.session(
    verify_connectivity=True,
    shared=True,
    shared_key="bootstrap-resilience",
    cluster_kwargs={"n_workers": 1, "threads_per_worker": 1, "processes": False, "dashboard_address": ":0"},
):
    dry_run_frame = await gateway.aload(
        global_track_id__in=[1, 2, 3, 4],
        columns=columns,
        return_type="dask",
        persist=True,
        resilient=True,
        diagnostics=True,
        dry_run=True,
    )
    resilient_frame = await gateway.aload(
        global_track_id__in=[1, 2, 3, 4],
        columns=columns,
        return_type="dask",
        persist=True,
        resilient=True,
        diagnostics=True,
    )
    persisted_frame = safe_persist(resilient_frame)
    safe_wait(persisted_frame, timeout=30)
    row_count = safe_compute(resilient_frame["global_track_id"].count())
    gathered_counts = safe_gather([resilient_frame["global_track_id"].count()])
    resilient_preview = safe_head(resilient_frame, n=3)
    async_persisted = await async_safe_persist(resilient_frame)
    await async_safe_wait(async_persisted, timeout=30)
    async_row_count = await async_safe_compute(resilient_frame["global_track_id"].count())
    async_preview = await async_safe_head(resilient_frame, n=2)
    async_gathered = await async_safe_gather([resilient_frame["global_track_id"].count()])
    del async_persisted
    del persisted_frame
    del resilient_frame

assert dry_run_frame.npartitions > 0
assert row_count > 0
assert async_row_count == row_count
assert gathered_counts == [row_count]
assert async_gathered == [row_count]
assert not resilient_preview.empty
assert len(async_preview) == 2

[2026-07-13 17:33:35][INFO][AsyncSqlDatabaseResource] Configured async select reflect_select=0.000s
[2026-07-13 17:33:35][INFO][AsyncSqlDatabaseResource] Build partitioned request elapsed=0.000s
[2026-07-13 17:33:37][INFO][AsyncSqlDatabaseResource] Partitioned SQL fast path: single partition rows=15633 ceiling=5000000 est_bytes_per_row=32 chunk_size=50000 total=2.901s planner_init=0.000s prepare_stmt=0.001s db_fetch=2.897s coerce=0.002s from_pandas=0.001s
[2026-07-13 17:33:37][INFO][AsyncSqlDatabaseResource] Configured async select reflect_select=0.000s
[2026-07-13 17:33:37][INFO][AsyncSqlDatabaseResource] Build partitioned request elapsed=0.000s
[2026-07-13 17:33:40][INFO][AsyncSqlDatabaseResource] Partitioned SQL fast path: single partition rows=15633 ceiling=5000000 est_bytes_per_row=32 chunk_size=50000 total=2.778s planner_init=0.000s prepare_stmt=0.000s db_fetch=2.776s coerce=0.001s from_pandas=0.001s


In [45]:
assert dask_is_probably_empty(result_dask) is False
assert dask_is_empty(result_dask) is False

unique_values = await UniqueValuesExtractor().extract_unique_values(
    result_dask,
    "product_type_id",
    "global_track_id",
    limit=10,
)

assert set(unique_values["product_type_id"]) == {1}
assert set(unique_values["global_track_id"]).issubset({5})
unique_values

{'product_type_id': [1], 'global_track_id': [5]}